# 03 - Erro com pontos flutuantes
Vamos aprender sobre como usar os erros de ponto flutuante para resolução de problemas numéricos.

Crie uma nova branch (versão) do repositório:

```bash
git branch semana3
```

Faça o checkout nessa nova branch:

```bash
git checkout semana3
```

<hr />

## Atividade 1
A função exponencial natural pode ser definida pelo limite:
$$
e^x=\lim_{n\to\infty}\left(1+\frac{x}{n}\right)^n,
$$
mas também é dada pela **série de Maclaurin**:
$$
e^x=\sum_{n=0}^{\infty}\frac{x^n}{n!}
=1+\frac{x}{1!}+\frac{x^2}{2!}+\frac{x^3}{3!}+\cdots
$$

Implemente em **Python** o cálculo de $e^x$ pela série, interrompendo a soma quando o termo ficar menor que o limite prático de contribuição, usando a precisão de máquina como critério.

In [ ]:
import math
import sys

def exp_series(x, atol=0.0):
    """ Aproxima e^x pela série de Maclaurin com critério de parada numérico. """
    eps = sys.float_info.epsilon
    s = 1.0
    term = 1.0
    n = 0
    tol_abs = max(atol, eps)

    while True:
        n += 1
        term *= x / n
        s += term
        if abs(term) < eps * abs(s) or abs(term) < tol_abs:
            break
        if n > 10_000:
            break
    return s, n, term

# Demonstração
for val in [1.0, 5.0, -2.0]:
    approx, nterms, last = exp_series(val)
    print(f"x={val:+g} -> e^x ≈ {approx:.16g} (math.exp={math.exp(val):.16g}, termos={nterms})")

## Atividade 2

Implemente:
$$
e^x\approx\left(1+\frac{x}{n}\right)^n
$$
com $n$ crescente, e:
1. Explique por que, para $x<0$ e $n$ muito grande, pode ocorrer **cancelamento catastrófico**;
2. Proponha um critério de parada numérico para encerrar o crescimento de $n$ sem perder precisão.

In [ ]:
def exp_limit(x, max_power=52):
    """Aproxima e^x pelo limite com avaliação estável de 1 + x/n."""
    eps = sys.float_info.epsilon
    n = max(1, math.ceil(abs(x)) + 1)
    previous = None

    for _ in range(max_power):
        increment = x / n
        if abs(increment) <= eps:
            break

        # log1p preserva os dígitos que seriam perdidos em 1 + x/n.
        current = math.exp(n * math.log1p(increment))
        if previous is not None and abs(current - previous) <= eps * max(1.0, abs(current)):
            return current, n
        previous = current
        n *= 2

    return previous, n // 2

for val in [-2.0, 1.0, 5.0]:
    approx, n = exp_limit(val)
    print(f"x={val:+g} -> limite ≈ {approx:.16g} (math.exp={math.exp(val):.16g}, n={n})")

print("Para x<0, 1 + x/n fica muito próximo de 1 quando n cresce.")
print("A forma literal perde dígitos significativos por cancelamento catastrófico.")
print("Usar log1p(x/n) evita essa perda; a comparação entre iterações encerra o processo quando a mudança fica menor que epsilon.")

## Atividade 3

Para $|x|$ grande, use:
$$
e^x = \left(e^{m\cdot 2^{-k}}\right)^{2^k}, \quad
k = \left\lceil \log_2\!\left(\frac{|x|}{\theta}\right)\right\rceil, \quad m = \frac{x}{2^k}
$$
Calcule $e^{m}$ pela série (Ex. 1) e depois eleve ao quadrado $k$ vezes.

In [ ]:
def exp_scaled(x, theta=1.0):
    """Calcula e^x reduzindo o argumento e fazendo quadraturas."""
    if x == 0.0:
        return 1.0, 0

    k = max(0, math.ceil(math.log2(abs(x) / theta)))
    reduced_x = x / (2**k)
    value, _, _ = exp_series(reduced_x)

    for _ in range(k):
        value *= value

    return value, k

for val in [20.0, -20.0, 50.0]:
    approx, k = exp_scaled(val)
    print(f"x={val:+g} -> e^x ≈ {approx:.16g} (math.exp={math.exp(val):.16g}, quadraturas={k})")

## Atividade 4

Use:
$$
\cos x=\sum_{n=0}^{\infty}(-1)^n\frac{x^{2n}}{(2n)!}
$$
com a recursão:
$$
t_{n+1}=t_n\cdot\frac{-x^2}{(2n+1)(2n+2)}
$$
Defina um critério de parada baseado em `epsilon` e compare o erro relativo para $x\in[-20,20]$ (200 pontos) contra `math.cos(x)`.

In [ ]:
def cos_series(x):
    """Aproxima cos(x) pela série usando recorrência entre termos."""
    eps = sys.float_info.epsilon
    total = 1.0
    term = 1.0
    n = 0

    while True:
        n += 1
        term *= -x * x / ((2 * n - 1) * (2 * n))
        candidate = total + term
        if abs(term) <= eps * max(1.0, abs(candidate)):
            return candidate, n
        total = candidate
        if n > 10_000:
            return total, n

xs = [(-20.0 + 40.0 * i / 199) for i in range(200)]
relative_errors = []
for x in xs:
    approx, _ = cos_series(x)
    reference = math.cos(x)
    scale = max(abs(reference), sys.float_info.epsilon)
    relative_errors.append(abs(approx - reference) / scale)

print(f"Maior erro relativo: {max(relative_errors):.3e}")
print(f"Erro relativo médio: {sum(relative_errors) / len(relative_errors):.3e}")

## Atividade 5

Dado $x$ e uma tolerância $\tau$, encontre o menor $N$ tal que:
$$
R_{N+1}(x)=\sum_{n=N+1}^{\infty}\frac{|x|^n}{n!} < \tau
$$

In [ ]:
def exponential_tail_bound(x, N):
    """Limite superior para R_(N+1), usando uma cauda geométrica."""
    a = abs(x)
    first_term = a ** (N + 1) / math.factorial(N + 1)
    ratio = a / (N + 2)
    if ratio >= 1.0:
        return math.inf
    return first_term / (1.0 - ratio)


def smallest_N_for_tail(x, tau):
    if tau <= 0:
        raise ValueError("A tolerância deve ser positiva.")

    N = 0
    while exponential_tail_bound(x, N) >= tau:
        N += 1
    return N, exponential_tail_bound(x, N)

for x, tau in [(1.0, 1e-12), (5.0, 1e-12), (20.0, 1e-12)]:
    N, bound = smallest_N_for_tail(x, tau)
    print(f"x={x:g}, tau={tau:.0e} -> N={N}, limite da cauda={bound:.3e}")

## Atividade 6

Usando `decimal` ou `mpmath`, compute $e^x$ em alta precisão e compare com o resultado de `float64` (Ex. 1) para $x\in\{20, 40, 50\}$.
Analise:
- perda de dígitos significativos;
- quando o `float64` começa a saturar por overflow.


In [ ]:
from decimal import Decimal, localcontext


def exp_high_precision(x, digits=80):
    with localcontext() as context:
        context.prec = digits
        return Decimal(str(x)).exp()


def significant_digits(reference, approximation):
    error = abs(reference - Decimal(str(approximation)))
    if error == 0:
        return math.inf
    return max(0, int(-math.log10(float(error / abs(reference)))))

for x in [20, 40, 50, 710, 711]:
    high_precision = exp_high_precision(x)
    try:
        float64_value = exp_series(float(x))[0]
        overflow = False
    except OverflowError:
        float64_value = math.inf
        overflow = True

    if math.isfinite(float64_value):
        digits = significant_digits(high_precision, float64_value)
        print(f"x={x}: float64={float64_value:.16g}, alta precisão={high_precision:.16g}, "
              f"dígitos corretos≈{digits}")
    else:
        print(f"x={x}: float64 saturou por overflow; alta precisão={high_precision:.16g}")

print(f"O maior x com e^x finito em float64 é aproximadamente log(max)={math.log(sys.float_info.max):.6f}.")

## Versionando o código

Submeta a branch para o servidor:

```bash
git add .
git commit -m "Semana 3"
git push origin semana3
```